In [ ]:
"""
sandbox_clusters.ipynb

Clustering neurons by their task-variable encoding 

Author: Stellina X. Ao
Created: 2026-07-22
Last Modified: 2026-07-22
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

## single sess

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
import numpy as np
from sg.models import Encoder, StrategyEncoder

encoder = Encoder(subj_id, sess_id, norm=True)
encoder.verify()

encoder_mb = StrategyEncoder(subj_id, sess_id, strategy_filter="mb")
encoder_mf = StrategyEncoder(subj_id, sess_id, strategy_filter="mf")

encoder_mb.verify()
encoder_mb.fit_encoder()

encoder_mf.verify()
encoder_mf.fit_encoder()

In [ ]:
# pick ideal num clusters, plot weight matrices for each cluster
# * only using tv weights

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

n_clusters = range(2, encoder.num_units // 2)
seeds = range(10)
silhouettes = np.zeros((len(n_clusters), len(seeds)))

for i, n in enumerate(n_clusters):
    for j, seed in enumerate(seeds):
        km = KMeans(n_clusters=n, random_state=seed)
        cluster_labels = km.fit_predict(encoder.encoder_weights[:, encoder.num_tents :])
        silhouettes[i][j] = silhouette_score(
            encoder.encoder_weights[:, encoder.num_tents :], cluster_labels
        )

n = n_clusters[np.argmax(silhouettes.mean(axis=1))]
print("cluster w/ best avg. silhouette score: ", n)

In [ ]:
silhouettes_mean = silhouettes.mean(axis=1)
silhouettes_std = silhouettes.std(axis=1)

plt.figure(tight_layout=True)
plt.plot(n_clusters, silhouettes_mean)
plt.fill_between(
    n_clusters,
    silhouettes_mean - silhouettes_std,
    silhouettes_mean + silhouettes_std,
    alpha=0.4,
)
plt.xlabel("n. clusters")
plt.ylabel("silhouette score (mean)")
plt.show()

In [ ]:
n = 5
km = KMeans(n_clusters=n, random_state=2)

# reg = "DLS"
km_labels = km.fit_predict(encoder.encoder_weights[:, encoder.num_tents :])

In [ ]:
n_cols = encoder.encoder_weights.shape[1] - encoder.num_tents
cluster_sizes = [(km_labels == i).sum() for i in range(n)]
max_rows = max(cluster_sizes)

px = 0.03  # inches per pixel — tune this to taste


for reg in encoder.regions:
    fig, axes = plt.subplots(
        ncols=n,
        nrows=1,
        figsize=(n * n_cols * px, max_rows * px),
    )

    for i, ax in enumerate(axes.flat):
        data = encoder.encoder_weights[
            np.intersect1d(np.where(km_labels == i)[0], encoder.reg_idxs[reg]),
            encoder.num_tents :,
        ]
        im = ax.imshow(data, vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")

        ax.set_box_aspect(
            data.shape[0] / n_cols
        )  # forces axes height ∝ n_rows, width unchanged
        ax.set_anchor("N")  # top-align so smaller clusters don't float to center
        ax.set_xticks([])
        ax.set_yticks([])

    fig.suptitle(reg)

## aggregate

In [ ]:
from sg.models import Encoder, StrategyEncoder
from core.data import subject_ids, session_ids
import numpy as np

weights = {subj_id: {} for subj_id in ["MR82", "MR83"]}

for subj_id in ["MR82", "MR83"]:
    weights[subj_id] = {k: {"DMS": [], "DLS": []} for k in ["full", "mb", "mf"]}
    for sess_id in [session_ids[np.where(subject_ids == subj_id)[0][0]][5]]:
        print(subj_id, sess_id)
        encoder = Encoder(subj_id, sess_id, norm=True)
        encoder_mb = StrategyEncoder(subj_id, sess_id, strategy_filter="mb")
        encoder_mf = StrategyEncoder(subj_id, sess_id, strategy_filter="mf")

        encoder.fit_encoder()

        try:
            encoder_mb.fit_encoder()
            encoder_mf.fit_encoder()
        except ValueError:
            continue

        if all(
            e.encoder_weights.shape[1] == 16 for e in [encoder, encoder_mb, encoder_mf]
        ):
            for reg in ["DMS", "DLS"]:
                weights[subj_id]["full"][reg].extend(
                    encoder.encoder_weights[encoder.reg_idxs[reg], encoder.num_tents :]
                )
                weights[subj_id]["mb"][reg].extend(
                    encoder_mb.encoder_weights[
                        encoder_mb.reg_idxs[reg], encoder_mb.num_tents :
                    ]
                )
                weights[subj_id]["mf"][reg].extend(
                    encoder_mf.encoder_weights[
                        encoder_mf.reg_idxs[reg], encoder_mf.num_tents :
                    ]
                )


weights = {
    subj_id: {
        k: {reg: np.array(v) for reg, v in weights[subj_id][k].items()}
        for k in weights[subj_id].keys()
    }
    for subj_id in ["MR82", "MR83"]
}

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


def get_n(ws):
    ws = np.vstack([ws_reg for ws_reg in ws.values()])
    n_clusters = range(2, 20)
    seeds = range(10)
    silhouettes = np.zeros((len(n_clusters), len(seeds)))

    for i, n in enumerate(n_clusters):
        for j, seed in enumerate(seeds):
            km = KMeans(n_clusters=n, random_state=seed)

            cluster_labels = km.fit_predict(ws)

            silhouettes[i][j] = silhouette_score(ws, cluster_labels)

    n = n_clusters[np.argmax(silhouettes.mean(axis=1))]
    print("cluster w/ best avg. silhouette score: ", n)

    return silhouettes, n


silhouettes = {
    subj_id: {k: get_n(ws) for k, ws in weights[subj_id].items()}
    for subj_id in ["MR82", "MR83"]
}

In [ ]:
def plot_silhouettes(n_clusters, silhouettes):
    silhouettes_mean = silhouettes.mean(axis=1)
    silhouettes_std = silhouettes.std(axis=1)

    fig, ax = plt.subplots(tight_layout=True)
    ax.plot(n_clusters, silhouettes_mean)
    ax.fill_between(
        n_clusters,
        silhouettes_mean - silhouettes_std,
        silhouettes_mean + silhouettes_std,
        alpha=0.4,
    )
    ax.set_xlabel("n. clusters")
    ax.set_ylabel("silhouette score (mean)")
    return ax


n_clusters = range(2, 20)
for subj_id in ["MR82", "MR83"]:
    for k, (sil, _) in silhouettes[subj_id].items():
        plot_silhouettes(n_clusters, sil)

In [ ]:
def plot_clusters_lines(subj_id, model):
    n = silhouettes[subj_id][model][1]
    km_labels = get_km_labels(n)

    colors = ["#2274a5", "#f1c40f", "#d90368"]

    for reg in ["DMS", "DLS"]:
        _, ax = plt.subplots()

        for i in range(n):
            cluster_idxs = np.where(km_labels[reg] == i)[0]

            for idx in cluster_idxs:
                ax.plot(weights[subj_id][model][reg][idx], alpha=0.5, color=colors[i])

In [ ]:
plot_clusters_lines("MR83", "full")

In [ ]:
def get_km_labels(n):
    ws = np.vstack([ws_reg for ws_reg in weights[subj_id][model].values()])

    km = KMeans(n_clusters=n, random_state=2).fit(ws)
    km_labels = {
        reg: km.predict(weights[subj_id][model][reg]) for reg in ["DMS", "DLS"]
    }

    return km_labels

In [ ]:
def plot_clusters(subj_id, model):
    n = silhouettes[subj_id][model][1]
    n_cols = 11

    km_labels = get_km_labels(n)

    cluster_sizes = [
        (km_labels[reg] == i).sum() for reg in ["DMS", "DLS"] for i in range(n)
    ]
    max_rows = max(cluster_sizes)
    px = 0.05

    # fpath = FIGURES_DIR / "lite" / subj_id / "bweight_clusters"

    for i, reg in enumerate(["DMS", "DLS"]):
        fig, axes = plt.subplots(
            ncols=n,
            nrows=1,
            figsize=(n * n_cols * px, max_rows * px),
        )

        for j in range(n):
            ax = axes[j]

            data = weights[subj_id][model][reg][np.where(km_labels[reg] == j)[0]]
            if data.shape[0] == 0:
                print(j, data.shape)
                continue
            ax.imshow(data, vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")

            print(data.shape[0], n_cols)
            ax.set_box_aspect(data.shape[0] / n_cols)
            ax.set_anchor("N")
            ax.set_xticks([])
            ax.set_yticks([])

            if j == 0:
                ax.set_title(reg, loc="left")

        fig.tight_layout()
        # save_fig(fig, fpath, f"{model}_{reg}.png")

    return fig, axes


model = "full"
for subj_id in ["MR82", "MR83"]:
    fig, axes = plot_clusters(subj_id, model)